In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [3]:
anime_data_client = AnimeDataClient(
    client_id,
    cache_file=PROJECT_ROOT / "anime_cache.json",
)

In [4]:
anime_data = anime_data_client.get_cache()

Build features

In [5]:
from anime_features import AnimeFeatureBuilder

builder = AnimeFeatureBuilder(
    anime_data,
    max_tfidf_features=3000,
    n_svd_components=300
)

anime_df = builder.build_features()

builder.svd_explained_variance

np.float64(0.41006426159563164)

Convert each anime in df to vectors

In [6]:
recommender = SimilarityRecommender()
anime_vectors = recommender.create_anime_vectors(anime_df)
anime_df_scaled = recommender.anime_df_scaled

Get user Data

In [7]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Bayesion Ridge Regression

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.linear_model import BayesianRidge
import numpy as np

rated_items, anime_df, anime_vectors, anime_df_scaled, builder = anime_data_client.get_rated_items(
    user_scores=user_scores,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
    anime_df=anime_df,
    anime_vectors=anime_vectors,
)

X = np.array([vec for _, vec, _ in rated_items])
y = np.array([score for _, _, score in rated_items], dtype=float)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=None
)

model = BayesianRidge()
model.fit(X_train, y_train)

pred, pred_std = model.predict(X_test, return_std=True)
pred = np.clip(pred, 1, 10)

mae = mean_absolute_error(y_test, pred)
rmse = root_mean_squared_error(y_test, pred)

baseline_pred = np.full_like(y_test, y_train.mean(), dtype=float)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"Baseline MAE: {baseline_mae:.3f}")
print(f"Improvement vs baseline: {baseline_mae - mae:.3f}")

MAE: 1.043
RMSE: 1.345
Baseline MAE: 1.442
Improvement vs baseline: 0.399


In [9]:
print("num ratings:", len(y))
print("mean:", y.mean())
print("median:", np.median(y))
print("std:", y.std())
print(pd.Series(y).value_counts().sort_index())

num ratings: 260
mean: 7.403846153846154
median: 7.0
std: 1.5696596321963387
1.0      1
3.0      2
4.0      8
5.0     10
6.0     57
7.0     57
8.0     48
9.0     59
10.0    18
Name: count, dtype: int64


Hit Rate

In [10]:
# Hide some high-rated anime, train on the rest, and see if the hidden likes show up near the top.
hit_rating_threshold = np.median(y) + 0.5 * np.std(y)
heldout_fraction = 0.25
top_ks = [5, 10, 20, 50, 100]

rated_eval = pd.DataFrame({
    "anime_id": [anime_id for anime_id, _, _ in rated_items],
    "score": [score for _, _, score in rated_items],
})

liked_eval = rated_eval[rated_eval["score"] >= hit_rating_threshold]
if len(liked_eval) < 2:
    raise ValueError("Need at least 2 high-rated anime to run a hit-rate holdout test.")

heldout_liked = liked_eval.sample(frac=heldout_fraction, random_state=None)
heldout_ids = set(heldout_liked["anime_id"])

train_eval = rated_eval[~rated_eval["anime_id"].isin(heldout_ids)]
train_ids = train_eval["anime_id"].tolist()

X_hit_train = anime_df_scaled.loc[train_ids].to_numpy()
y_hit_train = train_eval["score"].to_numpy(dtype=float)

hit_model = BayesianRidge()
hit_model.fit(X_hit_train, y_hit_train)

# Candidates are everything the model did not train on, including the hidden liked anime.
candidate_ids = [anime_id for anime_id in anime_df_scaled.index if anime_id not in set(train_ids)]
X_hit_candidates = anime_df_scaled.loc[candidate_ids].to_numpy()

hit_pred, hit_pred_std = hit_model.predict(X_hit_candidates, return_std=True)
hit_pred = np.clip(hit_pred, 1, 10)

title_by_id = {int(anime["id"]): anime["title"] for anime in anime_data.values()}
hit_recommendations = pd.DataFrame({
    "anime_id": candidate_ids,
    "title": [title_by_id.get(int(anime_id), "Unknown") for anime_id in candidate_ids],
    "actual_score": [user_scores.get(int(anime_id)) for anime_id in candidate_ids],
    "predicted_score": hit_pred,
    "uncertainty_raw": hit_pred_std,
})

uncertainty_clipped = hit_recommendations["uncertainty_raw"].clip(
    lower=hit_recommendations["uncertainty_raw"].quantile(0.05),
    upper=hit_recommendations["uncertainty_raw"].quantile(0.95),
)
if uncertainty_clipped.max() == uncertainty_clipped.min():
    hit_recommendations["uncertainty"] = 0.0
else:
    hit_recommendations["uncertainty"] = (
        (uncertainty_clipped - uncertainty_clipped.min())
        / (uncertainty_clipped.max() - uncertainty_clipped.min())
    )

hit_recommendations["ranking_score"] = (
    hit_recommendations["predicted_score"] - 4.5 * hit_recommendations["uncertainty"]
)
hit_recommendations["is_hidden_like"] = hit_recommendations["anime_id"].isin(heldout_ids)
hit_recommendations = hit_recommendations.sort_values("ranking_score", ascending=False)

hit_rows = []
for k in top_ks:
    top_k = hit_recommendations.head(k)
    hits = int(top_k["is_hidden_like"].sum())
    hit_rows.append({
        "k": k,
        "hits": hits,
        "heldout_likes": len(heldout_ids),
        "hit_rate": hits / len(heldout_ids),
        "precision_at_k": hits / k,
    })

hit_rate_results = pd.DataFrame(hit_rows)

print(f"Total rated anime: {len(rated_eval)}")
print(f"High-rated anime (score >= {hit_rating_threshold}): {len(liked_eval)}")
print(f"Hidden liked anime: {len(heldout_ids)}")
display(hit_rate_results)

# hit_recommendations[hit_recommendations["is_hidden_like"]].head(20)

Total rated anime: 260
High-rated anime (score >= 7.78482981609817): 125
Hidden liked anime: 31


,k,hits,heldout_likes,hit_rate,precision_at_k
0,5,3,31,0.096774,0.60
1,10,4,31,0.129032,0.40
2,20,5,31,0.161290,0.25
3,50,10,31,0.322581,0.20
4,100,16,31,0.516129,0.16


In [11]:
baseline_recommendations = hit_recommendations.copy()

# If "mean" is available in anime_df, rank by global MAL mean.
baseline_recommendations["baseline_score"] = anime_df.loc[
    baseline_recommendations["anime_id"], "mean"
].to_numpy()

baseline_recommendations = baseline_recommendations.sort_values(
    "baseline_score",
    ascending=False
)

baseline_rows = []
for k in top_ks:
    top_k = baseline_recommendations.head(k)
    hits = int(top_k["is_hidden_like"].sum())
    baseline_rows.append({
        "k": k,
        "baseline_hits": hits,
        "heldout_likes": len(heldout_ids),
        "baseline_hit_rate": hits / len(heldout_ids),
        "baseline_precision_at_k": hits / k,
    })

baseline_hit_rate_results = pd.DataFrame(baseline_rows)
hit_rate_results.merge(baseline_hit_rate_results, on=["k", "heldout_likes"])

,k,hits,heldout_likes,hit_rate,precision_at_k,baseline_hits,baseline_hit_rate,baseline_precision_at_k
0,5,3,31,0.096774,0.60,0,0.000000,0.00
1,10,4,31,0.129032,0.40,1,0.032258,0.10
2,20,5,31,0.161290,0.25,2,0.064516,0.10
3,50,10,31,0.322581,0.20,5,0.161290,0.10
4,100,16,31,0.516129,0.16,9,0.290323,0.09


Compare models

In [12]:
from anime_evaluation import HitRateEvaluator
 
n_runs = 50
compare_top_ks = (5, 10)
uncertainty_weight = 7.5

evaluator = HitRateEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    heldout_fraction=0.25,
)

compare_results, compare_summary = evaluator.compare_models(
    uncertainty_weight=uncertainty_weight,
    n_runs=n_runs,
    top_ks=compare_top_ks,
    include_ridge_cv=True,
    include_lasso=True,
    include_elastic_net=True,
    include_knn=True,
    knn_neighbors=20,
    knn_weights="distance",
    knn_metric="cosine",
    include_gradient_boosting=True,
    gradient_boosting_params={
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 2,
        "random_state": 42,
    },
    random_state=42,
)

display(compare_summary)

ridge_alpha_counts = (
    compare_results.dropna(subset=["ridge_alpha"])["ridge_alpha"]
    .value_counts()
    .sort_index()
)
display(ridge_alpha_counts)

lasso_alpha_counts = (
    compare_results.dropna(subset=["lasso_alpha"])["lasso_alpha"]
    .value_counts()
    .sort_index()
)
display(lasso_alpha_counts)

elastic_param_counts = (
    compare_results.dropna(subset=["elastic_alpha", "elastic_l1_ratio"])
    .groupby(["elastic_alpha", "elastic_l1_ratio"])
    .size()
    .sort_index()
)
elastic_param_counts

,model,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits
0,bayesian_ridge,5,0.528,0.179614,0.085161,0.028970,2.64
8,knn,5,0.288,0.181423,0.046452,0.029262,1.44
2,elastic_net_cv,5,0.172,0.121286,0.027742,0.019562,0.86
10,lasso_cv,5,0.172,0.114357,0.027742,0.018445,0.86
12,ridge_cv,5,0.172,0.140029,0.027742,0.022585,0.86
4,global_mean,5,0.116,0.134559,0.018710,0.021703,0.58
6,gradient_boosting,5,0.104,0.122824,0.016774,0.019810,0.52
1,bayesian_ridge,10,0.314,0.114304,0.101290,0.036872,3.14
9,knn,10,0.230,0.114731,0.074194,0.037010,2.30
13,ridge_cv,10,0.214,0.096911,0.069032,0.031262,2.14


ridge_alpha
8.497534      2
9.545485      2
12.045035     2
13.530478     2
15.199111     2
17.073526    10
19.179103    10
21.544347    10
24.201283    12
27.185882    24
30.538555    16
34.304693     6
38.535286     2
Name: count, dtype: int64

lasso_alpha
0.013895     6
0.017575     6
0.028118     2
0.035565    58
0.044984    28
Name: count, dtype: int64

elastic_alpha  elastic_l1_ratio
0.010985       0.1                  2
0.017575       0.7                  2
               0.9                  2
0.022230       0.5                  2
               0.7                  2
               0.9                  4
0.028118       0.3                  2
               0.5                  4
0.035565       0.9                 24
0.044984       0.9                 56
dtype: int64

## Results

Bayesian Ridge with uncertainty adjustment performed best across both cutoffs in the 50-run model comparison for user `chekkit`. This comparison uses the selected `all_features` representation, an uncertainty weight of **7.5**, and includes KNN as the strongest similarity-based baseline.

| Model | P@5 | P@10 | Avg hits@5 | Avg hits@10 |
| --- | ---: | ---: | ---: | ---: |
| bayesian_ridge | 0.528 | 0.314 | 2.64 | 3.14 |
| knn | 0.288 | 0.230 | 1.44 | 2.30 |
| ridge_cv | 0.172 | 0.214 | 0.86 | 2.14 |
| elastic_net_cv | 0.172 | 0.184 | 0.86 | 1.84 |
| lasso_cv | 0.172 | 0.184 | 0.86 | 1.84 |
| global_mean | 0.116 | 0.076 | 0.58 | 0.76 |
| gradient_boosting | 0.104 | 0.100 | 0.52 | 1.00 |

Conclusion: keep **Bayesian Ridge with uncertainty-aware ranking** as the recommendation reranker. It beats the KNN similarity baseline and the global mean baseline at both Precision@5 and Precision@10, and it also outperforms the raw linear and non-linear regressors in this comparison.

Bayesian Ridge vs KNN Significance Test

In [14]:
from anime_evaluation import HitRateEvaluator
 
n_runs = 500
compare_top_ks = (5, 10)
uncertainty_weight = 7.5

evaluator = HitRateEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    heldout_fraction=0.25,
)

compare_results, compare_summary = evaluator.compare_models(
    uncertainty_weight=uncertainty_weight,
    n_runs=n_runs,
    top_ks=compare_top_ks,
    include_ridge_cv=False,
    include_lasso=False,
    include_elastic_net=False,
    include_knn=True,
    knn_neighbors=20,
    knn_weights="distance",
    knn_metric="cosine",
    include_gradient_boosting=False,
    random_state=42,
)

display(compare_summary)

,model,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits
0,bayesian_ridge,5,0.5364,0.204643,0.086516,0.033007,2.682
4,knn,5,0.2852,0.177776,0.046000,0.028673,1.426
2,global_mean,5,0.0948,0.119673,0.015290,0.019302,0.474
1,bayesian_ridge,10,0.3246,0.128561,0.104710,0.041471,3.246
5,knn,10,0.2360,0.113357,0.076129,0.036567,2.360
3,global_mean,10,0.0638,0.058278,0.020581,0.018799,0.638


## Significance Test

The focused 500-run comparison tests whether **Bayesian Ridge with uncertainty-aware ranking** beats the **cosine KNN baseline** on the same train/holdout splits.

- Null hypothesis: Bayesian Ridge and KNN have the same mean Precision@K.
- Alternative hypothesis: Bayesian Ridge has higher mean Precision@K than KNN.

Because each run evaluates both models on the same split, this uses a paired one-sided test.

In [15]:
import pandas as pd
from scipy import stats

paired_precision = (
    compare_results[compare_results["model"].isin(["bayesian_ridge", "knn"])]
    .pivot_table(
        index=["run", "k"],
        columns="model",
        values="precision_at_k",
    )
    .dropna()
    .reset_index()
)

significance_rows = []

for k, group in paired_precision.groupby("k"):
    bayesian_precision = group["bayesian_ridge"].to_numpy()
    knn_precision = group["knn"].to_numpy()
    precision_diff = bayesian_precision - knn_precision

    t_test = stats.ttest_rel(
        bayesian_precision,
        knn_precision,
        alternative="greater",
    )
    wilcoxon_test = stats.wilcoxon(
        precision_diff,
        alternative="greater",
        zero_method="zsplit",
    )

    significance_rows.append({
        "k": k,
        "n_runs": len(group),
        "bayesian_precision": bayesian_precision.mean(),
        "knn_precision": knn_precision.mean(),
        "mean_difference": precision_diff.mean(),
        "paired_t_stat": t_test.statistic,
        "paired_t_p_value": t_test.pvalue,
        "wilcoxon_stat": wilcoxon_test.statistic,
        "wilcoxon_p_value": wilcoxon_test.pvalue,
    })

significance_results = pd.DataFrame(significance_rows)
significance_results

,k,n_runs,bayesian_precision,knn_precision,mean_difference,paired_t_stat,paired_t_p_value,wilcoxon_stat,wilcoxon_p_value
0,5,500,0.5364,0.2852,0.2512,20.851868,3.268397e-70,112686.5,1.072361e-54
1,10,500,0.3246,0.2360,0.0886,12.534947,7.893173e-32,98100.0,1.801206e-28


## Final Model Comparison Results

In the focused 500-run comparison, **Bayesian Ridge with uncertainty-aware ranking** clearly outperformed the **cosine KNN baseline** on the same train/holdout splits.

| Metric | Bayesian Ridge | Cosine KNN | Difference | Paired t-test p-value |
| --- | ---: | ---: | ---: | ---: |
| Precision@5 | 0.5364 | 0.2852 | +0.2512 | 3.27e-70 |
| Precision@10 | 0.3246 | 0.2360 | +0.0886 | 7.89e-32 |

With 500 paired runs, the paired t-test is appropriate for comparing the mean precision differences. The Wilcoxon signed-rank test also found the same conclusion, so the result is not dependent on only the t-test assumption.

Conclusion: the final Bayesian Ridge reranker is significantly better than the strongest KNN similarity baseline, especially at Precision@5.